# Retrieve Supabase Templates

Experiment to retrieve rows from the `templates` table in Supabase.

### Build Semantic doc (embedding_text)
yaml system

In [ ]:
# Building Template Profile semantic doc 
from agents.note_profiler.app.models import NoteProfile
from textwrap import dedent

def note_profile_to_embedding_text(profile: NoteProfile) -> str:
    """Serialize a NoteProfile into a deterministic text for embeddings."""

    def yaml_list(items: list[str]) -> str:
        return "\n".join(f"  - {item}" for item in items)

    parts = []
    
    # Map of output topic header to the profile attribute name
    topics = [
        ('name','name'),
        ('description','description'),
        ('instructions','instructions'),
        ('category','category'),
        ('tags','tags'),
        ('version','version'),
        # ('preview_markdown','preview_markdown'), #Example
        # ('template_markdown','template_markdown'), # Template Model
        ('purpose','purpose'),
        ('sections','sections'),
        ('organization_structure','organization_structure'),
        ('style','style'),
        ( 'raw_note_text','raw_note_text')
    ]
    
    for topic_name, attr_name in topics:
        if topic_name in profile:
            val = profile[attr_name]
            # Ensure the value exists, is not an empty string, and (if list) is not empty
            if val is not None and val != "" and (not isinstance(val, list) or len(val) > 0):
                formatted_list = yaml_list(val) if isinstance(val, list) else f"  - {val}"
                parts.append(f"{topic_name}:\n{formatted_list}")
                
    return "\n\n".join(parts).strip()


In [ ]:
note = NoteProfile # NoteProfiler output

# Add embedding text
note.embedding_text = note_profile_to_embedding_text(note) 

# Print or inspect the first result
print(note.embedding_text)


name:
  - Keep a Changelog

description:
  - This template is designed to meticulously document and track all modifications made to a project, product, or system across different versions. It offers a structured and chronological record of updates, making it an indispensable tool for understanding the evolution of a codebase or product. It's best suited for generating formal release notes, maintaining a transparent history of product changes, or providing detailed documentation of software updates. The template captures essential information such as version numbers, release dates, and systematically categorizes each change into distinct types like new features, improvements, bug fixes, removals, deprecations, and security patches. The information is organized hierarchically by release, with detailed bulleted lists under each change category, facilitating easy navigation and comprehension of what has been altered in each iteration. The style is objective, technical, and concise, ensurin

### Vectorize

In [ ]:
note.embedding_vector = response.embeddings[0].values

--- 
---


# Search Similarity and Retrival Engine


In [ ]:
# EXAMPLE Output from NoteProfiler (Input for RAG)
note_profile = NoteProfile(
    description= (
        "A rough technical planning note describing implementation ideas for a "
        "software project. The note mixes requirements, architectural decisions, "
        "implementation tasks and future ideas. Information is fragmented and "
        "would benefit from organization into logical sections while preserving "
        "all technical details."
    ),
    instructions="",
    raw_note_text="""
Need login pagee,Supabase auth,Google login maybe? (check poros andcons)
+email verification
pricing page
check RLS docs
migration after MVP (this will be a bitch, need to furher break down)
""",
    tags=[
        "technical",
        "software",
        "project",
        "planning",
        "implementation",
        "architecture",
        "requirements",
    ],
    purpose=[
        "plan implementation",
        "capture technical ideas",
        "track future work",
    ],
    sections=[
        "requirements",
        "authentication",
        "architecture",
        "implementation tasks",
        "future work",
    ],
    organization_structure=[
        "hierarchical sections",
        "group related concepts",
        "task lists",
        "logical ordering",
    ],
    style=[
        "technical",
        "concise",
        "fragmented",
        "markdown",
    ],
)

In [ ]:
query_text = note_profile_to_embedding_text(note_profile.model_dump()) 


In [ ]:
print(query_text)

tags:
  - technical
  - software
  - project
  - planning
  - implementation
  - architecture
  - requirements

purpose:
  - plan implementation
  - capture technical ideas
  - track future work

sections:
  - requirements
  - authentication
  - architecture
  - implementation tasks
  - future work

organization_structure:
  - hierarchical sections
  - group related concepts
  - task lists
  - logical ordering

style:
  - technical
  - concise
  - fragmented
  - markdown

raw_note_text:
  - 
Need login pagee,Supabase auth,Google login maybe? (check poros andcons)
+email verification
pricing page
check RLS docs
migration after MVP (this will be a bitch, need to furher break down)


In [ ]:
GEMINI_API_KEY = os.getenv("GEMINI_API_KEY")

from google import genai
from google.genai import types

client = genai.Client(api_key=GEMINI_API_KEY)

response = client.models.embed_content(
    model="gemini-embedding-001",
    contents=query_text,
    config=types.EmbedContentConfig(
        output_dimensionality=1536
    ),
)


In [17]:

# note_profile.embedding_vector = response.embeddings[0].values
embedding_vector = response.embeddings[0].values


---
---
## Running Supabase sql function match_templates

In [18]:
from supabase import create_client

supabase = create_client(
    SUPABASE_URL,
    SUPABASE_KEY,
)

results = (
    supabase
    .rpc(
        "match_templates",
        {
            "query_embedding": embedding_vector,   # list[float]
            "match_count": 5,
        },
    )
    .execute()
)

for template in results.data:
    print(template["name"], template["similarity"])

Architectural Decision Record 0.693203213522834
Zettelkasten 0.678610202224789
Keep a Changelog 0.6727800248987
Feynman Technique 0.665380246728926
SOAP Note 0.660417605174867


In [23]:
# setattr(note_profile,'embedding_vector', embedding_vector)
results.data

[{'id': 'adr',
  'name': 'Architectural Decision Record',
  'description': "This template is designed to capture and document significant technical and architectural decisions, their underlying context, the rationale for the chosen approach, and their anticipated consequences. It serves as a historical record for critical design choices, enabling teams to understand 'why' certain paths were taken and to track their long-term impact. It is best for documenting software architecture changes, resolving design dilemmas, ensuring team alignment on technical direction, and maintaining a clear log of key architectural evolution. The template contains information regarding the problem statement, influencing factors, the specific decision made, and both the positive and negative implications. Its structure logically progresses from problem definition to solution and impact assessment, often using bulleted lists for detailed consequences. The style is typically professional, objective, and analy